# Module 4.2: OCR Document Management & Full-Text Archiving with Paperless-ngx (`homelab-gpu:8000`)

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 4: File Management & Cloud De-Googling/Offboarding

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand how **Paperless-ngx** transforms static PDFs, scanned receipts, diplomas, tax filings, and migrated university Office documents (`.docx`, `.xlsx`, `.pptx`) into an OCR-indexed, instantaneously searchable personal document vault.
2. Architect Paperless-ngx cleanly across your **tiered storage**: fast **local NVMe** on `homelab-gpu` for PostgreSQL 17, Valkey, and the Whoosh search index, paired with **RAID-1 `UNAS 2` NFS** (`/mnt/unas_media/Paperless/`) for your archival PDFs and SMB-accessible `consume/` dropbox.
3. Deploy the full production stack (`paperless-ngx` + `postgres:17` + `valkey:8-alpine` + `apache/tika` + `gotenberg:8`) on `homelab-gpu:8000`.
4. Configure correspondents, document types, and automated tag matching rules, and test end-to-end OCR ingestion by dropping a file into your `UNAS 2` SMB `consume` folder.

## 4.2.1 Conceptual Background: Why Paperless-ngx Complements FileBrowser & Immich

While **FileBrowser Quantum** gives you a clean folder tree for general files (code repos, datasets, zip archives, ISOs), nested folders fail miserably when you are frantically searching 4 years later for *"that 2025 Purdue lab safety form"*, *"my 1098-T university tax PDF"*, or *"the warranty receipt for my UPS"*.

**Paperless-ngx** is the gold-standard open-source Document Management System (DMS):
- **Automatic Optical Character Recognition (OCRmyPDF / Tesseract):** Every scanned photo or flat PDF dropped into its `consume/` inbox is automatically OCR'd into a clean, text-selectable **PDF/A** archival copy (while keeping the byte-identical original untouched!).
- **Native Microsoft Office Ingestion (`Apache Tika` + `Gotenberg`):** Because you just migrated off a university OneDrive full of `.docx`, `.pptx`, and `.xlsx` files, enabling **Tika + Gotenberg** allows Paperless-ngx to parse, render to PDF, and full-text index Word, Excel, and PowerPoint files natively!
- **Zero Vendor Lock-In:** Every document is stored as a normal, human-readable PDF inside `/mnt/unas_media/Paperless/media/documents/archive/`—even if Docker disappears in 20 years, your PDFs remain 100% readable in any PDF viewer on your `UNAS 2`!
- **Automatic Local Machine Learning Tagging:** As you tag your first ~15–20 documents (`University`, `Taxes`, `Receipts`, `Career`, `Medical`), Paperless-ngx trains a lightweight local scikit-learn classifier right on `homelab-gpu` to auto-assign correspondents and tags to future documents!

```mermaid
flowchart LR
    Drop["Drop PDF / Scan / .docx<br>via SMB (UNAS 2) or Web Upload"] --> Consume["/mnt/unas_media/Paperless/consume<br>(NFSv3 Watch Folder on UNAS 2)"]
    Consume --> Web["Paperless-ngx Webserver<br>(homelab-gpu:8000 | OCRmyPDF)"]
    Web <--> Tika["Apache Tika + Gotenberg<br>(Office .docx/.xlsx/.pptx -> PDF)"]
    Web <--> DB["PostgreSQL 17 + Valkey + Search Index<br>(Fast Local NVMe on homelab-gpu)"]
    Web --> Archive["/mnt/unas_media/Paperless/media<br>(RAID 1 Redundant PDF/A Archive on UNAS 2)"]
```

## 4.2.2 Step-by-Step Laboratory Guide

### Task 4.2.1: Prepare Tiered Storage Directories & Compose Stack on `homelab-gpu`
Remember our Tiered Storage Rule: **PostgreSQL and Valkey stay on local NVMe (`~/homelab/stacks/paperless/`)**, while the durable PDF archive, `consume/` dropbox, and `export/` folder live on your **RAID-1 `UNAS 2`** (`/mnt/unas_media/Paperless/`)!

1. SSH into `homelab-gpu` (`ssh homelab-gpu`) and create both local DB directories and `UNAS 2` NFS document directories:
   ```bash
   mkdir -p ~/homelab/stacks/paperless/{data,pgdata,redisdata}
   mkdir -p /mnt/unas_media/Paperless/{media,consume,export}
   cd ~/homelab/stacks/paperless
   ```
2. Generate secure credentials in `docker-compose.env` (`chmod 600`):
   ```bash
   cat << EOT > ~/homelab/stacks/paperless/docker-compose.env
   PAPERLESS_DBENGINE=postgresql
   PAPERLESS_DBHOST=db
   PAPERLESS_DBNAME=paperless
   PAPERLESS_DBUSER=paperless
   PAPERLESS_DBPASS=$(openssl rand -hex 20)
   PAPERLESS_REDIS=redis://broker:6379
   PAPERLESS_SECRET_KEY=$(openssl rand -hex 32)
   PAPERLESS_TIME_ZONE=America/New_York
   PAPERLESS_OCR_LANGUAGE=eng
   PAPERLESS_URL=http://10.0.0.201:8000
   # Poll the NFS consume folder every 30s (inotify does not fire across NFS mounts!)
   PAPERLESS_CONSUMER_POLLING=30
   PAPERLESS_CONSUMER_RECURSIVE=true
   # Enable Office (.docx/.xlsx/.pptx) parsing via Apache Tika + Gotenberg
   PAPERLESS_TIKA_ENABLED=1
   PAPERLESS_TIKA_GOTENBERG_ENDPOINT=http://gotenberg:3000
   PAPERLESS_TIKA_ENDPOINT=http://tika:9998
   USERMAP_UID=$(id -u)
   USERMAP_GID=$(id -g)
   EOT
   chmod 600 ~/homelab/stacks/paperless/docker-compose.env

   # Also expose POSTGRES_PASSWORD matching PAPERLESS_DBPASS for the Postgres container via .env
   grep '^PAPERLESS_DBPASS=' ~/homelab/stacks/paperless/docker-compose.env \
       | sed 's/^PAPERLESS_DBPASS=/POSTGRES_PASSWORD=/' > ~/homelab/stacks/paperless/.env
   chmod 600 ~/homelab/stacks/paperless/.env
   ```
   *(Crucial Linux/NFS Engineering Detail: Notice **`PAPERLESS_CONSUMER_POLLING=30`**! Linux kernel `inotify` events only work on local filesystems—not across NFSv3 mounts. Setting 30-second polling ensures that any PDF you drag into `smb://10.0.0.202/MediaArchive/Paperless/consume` from your Mac/Windows laptop is picked up and OCR'd automatically within 30 seconds!)*

3. Create `~/homelab/stacks/paperless/docker-compose.yml`:
   ```bash
   cat << 'EOT' > ~/homelab/stacks/paperless/docker-compose.yml
   name: paperless_ngx

   services:
     broker:
       image: docker.io/valkey/valkey:8-alpine
       container_name: paperless_broker
       restart: unless-stopped
       volumes:
         - ./redisdata:/data

     db:
       image: docker.io/library/postgres:17-alpine
       container_name: paperless_db
       restart: unless-stopped
       volumes:
         - ./pgdata:/var/lib/postgresql/data
       environment:
         POSTGRES_DB: paperless
         POSTGRES_USER: paperless
         POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
       healthcheck:
         test: ["CMD-SHELL", "pg_isready -U paperless -d paperless"]
         interval: 10s
         timeout: 5s
         retries: 5

     tika:
       image: docker.io/apache/tika:latest
       container_name: paperless_tika
       restart: unless-stopped

     gotenberg:
       image: docker.io/gotenberg/gotenberg:8
       container_name: paperless_gotenberg
       restart: unless-stopped
       command:
         - "gotenberg"
         - "--chromium-disable-javascript=true"
         - "--chromium-allow-list=file:///tmp/.*"

     webserver:
       image: ghcr.io/paperless-ngx/paperless-ngx:latest
       container_name: paperless_webserver
       restart: unless-stopped
       depends_on:
         db:
           condition: service_healthy
         broker:
           condition: service_started
         gotenberg:
           condition: service_started
         tika:
           condition: service_started
       ports:
         - "8000:8000"
       volumes:
         # Fast local NVMe search index & ML classifier data:
         - ./data:/usr/src/paperless/data
         # Redundant RAID-1 UNAS 2 storage for archived PDFs, inbox, and exports:
         - /mnt/unas_media/Paperless/media:/usr/src/paperless/media
         - /mnt/unas_media/Paperless/export:/usr/src/paperless/export
         - /mnt/unas_media/Paperless/consume:/usr/src/paperless/consume
       env_file:
         - docker-compose.env
   EOT
   ```
4. Pull images, launch the stack, and create your initial Paperless-ngx superuser account:
   ```bash
   cd ~/homelab/stacks/paperless
   docker compose pull
   docker compose up -d
   # Wait ~20 seconds for DB migrations, then create your admin login:
   docker compose exec webserver createsuperuser
   ```

### Task 4.2.2: Test Live OCR Document Ingestion via `UNAS 2` SMB Inbox
1. Open your browser and log into Paperless-ngx at:
   ```text
   http://10.0.0.201:8000
   ```
2. Create starter **Tags** (`University`, `Diploma & Transcripts`, `Taxes`, `Career`, `Receipts`, `Homelab`) under **Manage $\rightarrow$ Tags**.
3. **Test End-to-End SMB $\rightarrow$ NFS $\rightarrow$ OCR Pipeline:**
   - From your laptop (via SMB `smb://10.0.0.202/MediaArchive/Paperless/consume`) or directly in the Paperless web UI, drop a sample university PDF or `.docx` document (e.g., a syllabus, transcript, or receipt) into the `consume` folder!
   - Watch live OCR processing on `homelab-gpu`:
     ```bash
     docker logs -f paperless_webserver
     ```
   - Within ~30 seconds, the file disappears from `consume/`, is OCR-converted into a searchable PDF/A inside `/mnt/unas_media/Paperless/media/documents/archive/`, and appears in your Paperless-ngx dashboard!
4. Click the search bar at the top of Paperless-ngx and search for a specific phrase buried inside page 2 of your PDF—watch it return instantaneously with highlighted matches!

### 🖼️ Task 4.2.2 Verification: Paperless-ngx OCR Vault (`10.0.0.201:8000`)

**Verification Objective:** Capture a screenshot of **Paperless-ngx** (`http://10.0.0.201:8000`) displaying your first ingested & OCR-indexed document (alongside `docker compose ps` showing all 5 Paperless containers running cleanly).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Paperless-ngx + PostgreSQL 17 + Valkey + Apache Tika + Gotenberg deployed on `homelab-gpu:8000`.
- [ ] Database & search index located on fast local NVMe (`~/homelab/stacks/paperless/`), while `media/`, `consume/`, and `export/` map cleanly to `UNAS 2` RAID 1 (`/mnt/unas_media/Paperless/`).
- [ ] `PAPERLESS_CONSUMER_POLLING=30` active so files dropped via `UNAS 2` SMB are automatically ingested and OCR'd.
- [ ] Live OCR ingestion and full-text search verified on a test document.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 5.1: Open-Source Email Archiving & `.edu` Mailbox Preservation with Mail-Archiver](../05_email_archiving_and_preservation/05_1_open_source_email_archiving_mail_archiver.ipynb).